# Ball detector training on Colab - far-court focus (T4)

Separate from `train_ball_detector_colab.ipynb` - this run targets the far-court miss gap specifically (small, distant ball is much easier for the detector to miss than a near-court one), rather than being a general re-fine-tune. Two changes from that notebook:

1. **`--imgsz 1664`** (vs the script's 1280 default) - https://github.com/s-ganguli/AI-Tennis-Ball-Bounce-Detection found meaningfully better far-court recall at this resolution, since the ball is only a handful of pixels wide at the far baseline and more resolution directly helps see it.
2. **An additional far-court hard-positive bundle**, merged into the training set on top of the base dataset - frames the CURRENT detector missed entirely at the far end of the court, hand-relabeled with the true ball position (see `scripts/extract_ball_miss_candidates.py` + `scripts/add_hard_positives.py`, same pattern already used for the net-post false-positive fix). This is the part that actually teaches the model to recognize a far-court ball, not just gives it more pixels to work with - resolution alone only helps up to a point if the training data still skews near-court.

**Before running:** upload BOTH of these to your Google Drive:

- `tennis_ball_dataset_colab.zip` (same base dataset bundle `train_ball_detector_colab.ipynb` uses - `data/raw/train`, `data/raw/valid`, current `weights/ball_detector.pt`, `configs/ball_dataset.yaml`) at `My Drive/tennis_ball_dataset_colab.zip`
- `farcourt_hard_positives.zip` at `My Drive/farcourt_hard_positives.zip` - build it locally with:
  ```
  python scripts/add_hard_positives.py --input your_video.mp4 \
      --labels outputs/your_video/ball_misses_labeled.csv \
      --out /tmp/farcourt_bundle --prefix farcourt
  cd /tmp/farcourt_bundle && zip -r farcourt_hard_positives.zip images labels
  ```
  (`--out` pointed at a fresh `/tmp` directory, not your live `data/raw/train` - that way the zip has ONLY the new far-court examples, not your whole existing dataset)

If you don't have the second bundle yet, run the cells anyway - it skips the merge step and trains at the higher resolution on the base dataset alone, still a real (if smaller) far-court improvement on its own. Add the hard-positive bundle and re-run later for the full effect.

Then: Runtime -> Change runtime type -> T4 GPU, and run the cells in order.

Checkpoints save straight to Drive (`--project` points at Drive), so a runtime disconnect (free tier: ~90min idle timeout, ~12h hard cap) doesn't lose progress - re-run the training cell pointing `--model` at the saved `last.pt` to keep fine-tuning from there.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the base dataset bundle, and the far-court hard-positive bundle if present
!mkdir -p /content/data_bundle
!unzip -q "/content/drive/MyDrive/tennis_ball_dataset_colab.zip" -d /content/data_bundle

import os
farcourt_zip = "/content/drive/MyDrive/farcourt_hard_positives.zip"
HAS_FARCOURT_BUNDLE = os.path.exists(farcourt_zip)
if HAS_FARCOURT_BUNDLE:
    !mkdir -p /content/farcourt_bundle
    !unzip -q "{farcourt_zip}" -d /content/farcourt_bundle
    !find /content/farcourt_bundle -maxdepth 3 -type d
else:
    print("No farcourt_hard_positives.zip found on Drive - training on the base dataset only "
          "(still gets the imgsz=1664 benefit, just not the new hard-positive examples).")

In [ ]:
# Clone the repo (public GitHub) at the working branch
!git clone -b claude/tennis-ball-yolo-tracking-p8ntwh --depth 1 https://github.com/will-wang1/tennis-tracking-yolo-v1.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
# Drop the unzipped dataset/checkpoint/config into the repo's expected layout,
# then merge in the far-court hard positives (train split only - they're extra
# training signal, not held-out validation frames) if the bundle was found above.
!rm -rf /content/repo/data/raw/train /content/repo/data/raw/valid
!mkdir -p /content/repo/data/raw
!cp -r /content/data_bundle/data/raw/train /content/repo/data/raw/train
!cp -r /content/data_bundle/data/raw/valid /content/repo/data/raw/valid
!cp /content/data_bundle/weights/ball_detector.pt /content/repo/weights/ball_detector.pt
!cp /content/data_bundle/configs/ball_dataset.yaml /content/repo/configs/ball_dataset.yaml

if HAS_FARCOURT_BUNDLE:
    !cp /content/farcourt_bundle/images/*.jpg /content/repo/data/raw/train/images/
    !cp /content/farcourt_bundle/labels/*.txt /content/repo/data/raw/train/labels/

!ls /content/repo/data/raw/train/images | wc -l
!ls /content/repo/data/raw/valid/images | wc -l

In [ ]:
# Train at imgsz=1664 for far-court recall. batch lowered from the other
# notebook's 16 to 8 - 1664 res uses a lot more VRAM per image than 1280
# does, and 8 still fits the T4's 16GB comfortably.
# --project writes straight to Drive so checkpoints survive a runtime disconnect.
!mkdir -p /content/drive/MyDrive/tennis_colab/runs
!python scripts/train.py \
    --model weights/ball_detector.pt \
    --imgsz 1664 \
    --epochs 40 \
    --batch 8 \
    --patience 15 \
    --name ball_detector_farcourt_colab \
    --workers 2 \
    --project /content/drive/MyDrive/tennis_colab/runs

## After training

The best checkpoint is at `My Drive/tennis_colab/runs/ball_detector_farcourt_colab/weights/best.pt` - already in Drive, no download-from-Colab step needed. On your local machine:

1. Download `best.pt` from that Drive folder.
2. `cp weights/ball_detector.pt weights/ball_detector_pre_farcourt_backup.pt` (back up the current one first)
3. Move the downloaded `best.pt` to `weights/ball_detector.pt`
4. Since this was trained at `imgsz=1664`, pass `--imgsz 1664` to `main.py` too when using it - a mismatched inference resolution gives up most of the far-court benefit this was trained for.
5. `python -m pytest tests/ -q` to sanity check nothing broke.